# Chapter 3: Language Conditioning
Swap the lookup-table language encoder for a real LM (SmolLM2) and compare concat vs cross-attention fusion on 11 instructions.

In [ ]:
!pip install torch torchvision numpy gymnasium matplotlib transformers

In [ ]:
# Skips the clone if it is already present, and surfaces the real error if
# it fails, rather than hiding it and failing confusingly on the %cd below.
![ -d vla-from-scratch ] || git clone https://github.com/FanFeast/vla-from-scratch.git
%cd vla-from-scratch/chapters/03_language_conditioning

## The Instruction Set

Chapter 2 had one implicit task. Here the policy must follow **11 canonical instructions** across 4 behavior categories, and generalize to **paraphrases it never saw during training** -- the metric that separates a real language encoder from a lookup table.

In [ ]:
from mini_pusht import MiniPushT, ScriptedExpert, CANONICAL_INSTRUCTIONS, PARAPHRASES
import matplotlib.pyplot as plt

for i, instr in enumerate(CANONICAL_INSTRUCTIONS):
    print(f"{i:2d}. {instr}")

print(f"\nParaphrases (held out of training):")
for canonical, paras in list(PARAPHRASES.items())[:3]:
    print(f"  {canonical!r} -> {paras}")

## Same Scene, Different Instruction

The observation is identical; only the instruction changes. A vision-only policy cannot tell these apart -- language is the only signal that disambiguates them.

In [ ]:
env = MiniPushT(size=224)

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, instr in zip(axes, ["push block to goal", "push block left", "move agent up"]):
    obs, info = env.reset(seed=7, options={"instruction": instr})
    ax.imshow(obs)
    ax.set_title(instr, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Collect Demos

`collect_demos` balances episodes across all 11 canonical instructions, so no behavior is under-represented. We use the **quick preset** (64x64, 220 episodes) so this runs in minutes on a Colab T4 -- see the README for full-scale numbers.

In [ ]:
from cross_attention import collect_demos, PushTDataset, PRESETS

preset = PRESETS["quick"]      # {"size": 64, "n_demos": 220, "epochs": 10}
print(f"preset: {preset}")

env = MiniPushT(size=preset["size"])
expert = ScriptedExpert(size=preset["size"])
demos = collect_demos(env, expert, n_episodes=preset["n_demos"])
dataset = PushTDataset(demos)

print(f"Collected {len(demos)} transitions from {preset['n_demos']} episodes")
print(f"Image shape: {demos[0]['image'].shape}")
print(f"Instruction of first transition: {demos[0]['instruction']!r}")

## Train the Four Configurations

| Config | Language encoder | Fusion |
|---|---|---|
| Lookup + Concat | embedding table | concat |
| SmolLM2-135M + Concat | frozen SmolLM2-135M, mean-pooled | concat |
| SmolLM2-135M + CrossAttn | frozen SmolLM2-135M, per-token | cross-attention |
| SmolLM2-360M + CrossAttn | frozen SmolLM2-360M, per-token | cross-attention |

Cross-attention is **much** slower per epoch (vision attends to every language token). On the quick preset that is tolerable; at full scale it is ~40x.

In [ ]:
import time
import torch
from cross_attention import build_vla, train, evaluate, CONFIGS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

results = {}
losses_dict = {}
models = {}

for cfg in CONFIGS:
    name = cfg["name"]
    print(f"\n{'='*50}\nTraining: {name}\n{'='*50}")

    model = build_vla(fusion_type=cfg["fusion"], lm_type=cfg["lm"])
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {n_train:,}")

    start = time.time()
    losses = train(model, dataset, epochs=preset["epochs"], device=device)
    elapsed = time.time() - start

    success = evaluate(model, env, n_episodes=55, device=device)
    para = evaluate(model, env, n_episodes=55, device=device, use_paraphrases=True)

    results[name] = {"params": n_train, "time": elapsed,
                     "loss": losses[-1], "success": success, "para": para}
    losses_dict[name] = losses
    models[name] = model

print(f"\n{'='*50}\nRESULTS\n{'='*50}")
for name, r in results.items():
    print(f"{name:<28} {r['params']:>8,} params  {r['time']:>7.1f}s  "
          f"loss={r['loss']:.3f}  success={r['success']*100:5.1f}%  paraphrase={r['para']*100:5.1f}%")

## Training Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, losses in losses_dict.items():
    ax.plot(range(1, len(losses) + 1), losses, marker="o", markersize=3, label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training loss by language-conditioning strategy")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Canonical vs Paraphrase: the Real Test

Both bars use the same trained policy. The left bar is instructions it trained on; the right bar is rewordings it has never seen. The lookup encoder has no way to map an unseen string to a behavior -- it collapses to guessing from vision alone.

In [ ]:
import numpy as np

names = list(results.keys())
x = np.arange(len(names))
w = 0.38

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, [results[n]["success"]*100 for n in names], w, label="Canonical")
ax.bar(x + w/2, [results[n]["para"]*100 for n in names], w, label="Paraphrase")
ax.set_ylabel("Success rate (%)")
ax.set_title("Instruction following: seen vs unseen phrasings")
ax.set_xticks(x)
ax.set_xticklabels([n.replace(" + ", "\n+ ") for n in names], fontsize=8)
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## Per-Instruction Breakdown

Aggregate success hides which behaviors are hard. `agent_move` instructions are usually easiest (ignore the block entirely); `push_loc` is hardest (two-phase: reach the block, then push it to a specific coordinate).

In [ ]:
from cross_attention import evaluate_per_instruction

best = max(results, key=lambda n: results[n]["success"])
print(f"Breaking down: {best}\n")

per_instr = evaluate_per_instruction(models[best], env, n_per_instr=10, device=device)

order = sorted(per_instr, key=per_instr.get)
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(range(len(order)), [per_instr[k]*100 for k in order])
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=8)
ax.set_xlabel("Success rate (%)")
ax.set_title(f"Per-instruction success -- {best}")
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## What We Learned

A **pretrained language model is the biggest lever** -- it maps unseen paraphrases into roughly the same embedding region as the canonical instruction, which a lookup table can never do.

**Concat beat cross-attention here**, and that is worth sitting with. Cross-attention lets each vision token attend to individual words, which matters when the instruction refers to *one object among many*. MiniPushT has one block and five actions, so there is nothing to ground -- the extra machinery only adds cost. At full scale (README): 89.1% success / 40.0% paraphrase for SmolLM2+Concat, versus 81.8% / 38.2% for cross-attention at 40x the training time.

Cross-attention earns its keep from Chapter 5 onward, once scenes contain multiple objects and instructions get specific.

**Next:** Chapter 4 leaves language alone and asks how to *represent actions* -- discrete tokens vs regression vs diffusion.